# Smart MCQ Solver — 6-Model Comparison

Predict the **top-3 answers** for 5-option multiple-choice questions. Metric: **MAP@3**.

## Models

| # | Model | Type | Requirement |
|---|---|---|---|
| 1 | TF-IDF + cosine similarity | unsupervised baseline | — |
| 2 | **Sentence-transformer cosine (MPNet)** | pre-trained, zero-shot | **pre-trained model** |
| 3 | Deepnet over sentence-pair features | supervised, 5-fold | model of choice |
| 4 | TF-IDF + Deepnet | supervised, 5-fold | model of choice |
| 5 | **BiLSTM + Attention** | supervised, no pre-trained weights | **model from scratch** |
| 6 | DeBERTa-v3 fine-tune | supervised (additional experiment) | — |

## Model 2 architecture (the core idea)

```
        Question                          Options A-E
            |                                  |
            v                                  v
   MPNet Sentence Transformer  <---same--->  MPNet
            |                                  |
            v                                  v
   Question Embedding (768)         5 Option Embeddings (768)
            |                                  |
            +----------------+-----------------+
                             v
                    Cosine Similarity
                             v
                    5 Similarity Scores
                             v
                  Rank Highest -> Lowest
                             v
                       Top-3 Answers
```

The question is compared **directly to each option**. No training corpus is
consulted, so performance on validation and on the hidden test set measure the
same thing — unlike retrieval, which only works when a similar question already
exists in training.

## Evaluation

Every model reports **MAP@3, Accuracy, Macro F1 and Weighted F1**, measured
out-of-fold for the trained models so the comparison is honest.

**Baselines to beat:** random ordering = 0.3667; always answering the three most
frequent letters = **0.4213**.


In [ ]:
import os, re, gc, math, random, warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Config — CPU only
# ---------------------------------------------------------------------------
BASE        = "/kaggle/input/competitions/smart-mcq-solver-challenge"
TRAIN_PATH  = f"{BASE}/train.csv"
TEST_PATH   = f"{BASE}/test.csv"
OUTPUT_PATH = "/kaggle/working/submission.csv"

OPTIONS  = ["A", "B", "C", "D", "E"]
SEED     = 42
VAL_SIZE = 0.20
N_FOLDS  = 5

MPNET_ID = "sentence-transformers/all-mpnet-base-v2"

# DeBERTa is kept as an experiment only. It is a 435M model: fine-tuning it on
# CPU is not practical, so it stays off unless a GPU is attached.
RUN_DEBERTA = torch.cuda.is_available()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()
print(f"torch {torch.__version__} | device: {DEVICE}")
print(f"DeBERTa experiment: {'ON' if RUN_DEBERTA else 'OFF (no GPU — expected on CPU)'}")


## 1. Load data

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {train_df.shape}")
print(f"Test  : {test_df.shape}")

counts = train_df["answer"].value_counts().reindex(OPTIONS)
probs  = counts / counts.sum()
order  = list(probs.sort_values(ascending=False).index)

PRIOR_MAP3 = probs[order[0]] + probs[order[1]] / 2 + probs[order[2]] / 3
LABEL_LOGPRIOR = np.log(probs[OPTIONS].values.astype(np.float64))

print("\nAnswer distribution:")
for o in order:
    print(f"  {o}  {counts[o]:>4}  ({probs[o]:.1%})")
print(f"\nRandom-ordering MAP@3        : 0.3667")
print(f"Always '{' '.join(order[:3])}' MAP@3          : {PRIOR_MAP3:.4f}   <- the bar to beat")

train_df.head(3)


## 2. Preprocessing and split

Prompts carry boilerplate prefixes ("Pick the best possible answer:", …) which
are stripped. **The split is made on the CLEANED prompt**, because two prompts
that differ only by prefix become identical after cleaning — splitting on raw
text would put the same question on both sides and make validation meaningless.


In [ ]:
BOILERPLATE = [
    r"^Pick the best possible answer:\s*", r"^Choose the correct answer:\s*",
    r"^Select the most accurate option:\s*", r"^Identify the correct statement:\s*",
    r"^Determine the correct option:\s*", r"^Which of the following\s*",
    r"\s*among the listed options\.?$", r"\s*from the following choices\.?$",
    r"\s*based on the given context\.?$", r"\s*carefully\.?$",
]


def clean_text(t):
    t = re.sub(r"\s+", " ", str(t)).strip()
    for p in BOILERPLATE:
        t = re.sub(p, "", t, flags=re.IGNORECASE).strip()
    return t


def clean_frame(df):
    out = df.copy()
    for c in ["prompt"] + OPTIONS:
        out[c] = out[c].map(clean_text)
    return out


train = clean_frame(train_df)
test  = clean_frame(test_df)

n_raw, n_clean = train_df["prompt"].nunique(), train["prompt"].nunique()
print(f"Unique prompts  raw {n_raw}  ->  cleaned {n_clean}   "
      f"({n_raw - n_clean} collapsed by preprocessing)")

uniq = train["prompt"].unique()
tr_p, va_p = train_test_split(uniq, test_size=VAL_SIZE, random_state=SEED)

train_split = train[train["prompt"].isin(tr_p)].reset_index(drop=True)
val_split   = train[train["prompt"].isin(va_p)].reset_index(drop=True)
assert not (set(train_split["prompt"]) & set(val_split["prompt"]))

y_val   = val_split["answer"].tolist()
y_train = train["answer"].tolist()

print(f"Train {len(train_split)} | Val {len(val_split)} | no cleaned-prompt overlap")


## 3. Metrics — MAP@3, Accuracy, Macro F1

In [ ]:
def average_precision_at_3(actual, predicted):
    for rank, p in enumerate(predicted[:3], start=1):
        if p == actual:
            return 1.0 / rank
    return 0.0


def map_at_3(actuals, preds):
    return float(np.mean([average_precision_at_3(a, p) for a, p in zip(actuals, preds)]))


def top3(scores):
    """(n,5) score matrix -> list of top-3 letter lists."""
    return [[OPTIONS[i] for i in np.argsort(-s)][:3] for s in np.asarray(scores)]


def evaluate(name, actuals, scores, store=None):
    preds = top3(scores)
    t1 = [p[0] for p in preds]
    m = {
        "MAP@3":    map_at_3(actuals, preds),
        "Accuracy": accuracy_score(actuals, t1),
        "MacroF1":  f1_score(actuals, t1, labels=OPTIONS, average="macro", zero_division=0),
        "WgtF1":    f1_score(actuals, t1, labels=OPTIONS, average="weighted", zero_division=0),
    }
    print(f"{name:<28} MAP@3 {m['MAP@3']:.4f} | Acc {m['Accuracy']:.4f} | "
          f"MacroF1 {m['MacroF1']:.4f} | WgtF1 {m['WgtF1']:.4f}")
    if store is not None:
        store[name] = m
    return m


RESULTS = {}
print(f"reference: random 0.3667 | prior {PRIOR_MAP3:.4f}")


## Model 1 — TF-IDF + cosine similarity  *(baseline)*

Each option is scored by the TF-IDF cosine similarity between the **question**
and the **option text**. Purely lexical: it rewards word overlap, so it fails
whenever the correct answer paraphrases the question instead of repeating it.


In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True,
                        strip_accents="unicode", min_df=1)
# fit on all text (no labels used) so the test vocabulary is represented
tfidf.fit(pd.concat([train["prompt"], test["prompt"]] +
                    [train[o] for o in OPTIONS] + [test[o] for o in OPTIONS]).astype(str))


def tfidf_scores(df):
    q = tfidf.transform(df["prompt"].astype(str))
    out = np.zeros((len(df), len(OPTIONS)), dtype=np.float32)
    for j, o in enumerate(OPTIONS):
        opt = tfidf.transform(df[o].astype(str))
        out[:, j] = np.asarray(q.multiply(opt).sum(axis=1)).ravel() / (
            (np.sqrt(q.multiply(q).sum(axis=1)).A.ravel() *
             np.sqrt(opt.multiply(opt).sum(axis=1)).A.ravel()) + 1e-9)
    return out


tfidf_val   = tfidf_scores(val_split)
tfidf_train = tfidf_scores(train)
tfidf_test  = tfidf_scores(test)

evaluate("1. TF-IDF cosine", y_val, tfidf_val, RESULTS)


## Model 2 — MPNet sentence transformer  *(pre-trained)*

```
   Question --> MPNet --> question embedding (768)
   Options  --> MPNet --> 5 option embeddings (768)
                   |
            cosine similarity
                   |
        rank high -> low --> top 3
```

Embeddings are L2-normalised, so the cosine reduces to a dot product. Nothing is
trained and no corpus is consulted — the model is asked directly which option is
closest in meaning to the question.


In [ ]:
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------------------------
# Encoder choice matters more than anything else here, and it is free on CPU.
#
# all-mpnet-base-v2 is trained for SYMMETRIC similarity ("are these two
# sentences alike?"). This task is ASYMMETRIC -- "does this option ANSWER this
# question?" -- which is what the multi-qa-* models were trained on (they are
# fine-tuned on 215M question/answer pairs). For question->answer matching that
# is usually a large gain over a general-purpose encoder.
#
# Each is evaluated on its own, then all are combined. Trim this list if the
# session is running long; the first entry alone is a reasonable default.
# ---------------------------------------------------------------------------
ENCODERS = [
    "sentence-transformers/multi-qa-mpnet-base-cos-v1",   # QA-tuned  <- expect best
    "sentence-transformers/all-mpnet-base-v2",            # general similarity
    "BAAI/bge-small-en-v1.5",                             # small + strong, fast on CPU
]

ENC = {}          # name -> dict(q_train, o_train, q_test, o_test, scores...)


def embed_with(model, texts, bs=64):
    return normalize(model.encode(list(texts), batch_size=bs,
                                  show_progress_bar=False, convert_to_numpy=True))


def run_encoder(name):
    print(f"\n--- {name} ---")
    m = SentenceTransformer(name, device=str(DEVICE))
    d = m.get_sentence_embedding_dimension()

    def enc(df):
        q = embed_with(m, df["prompt"].astype(str))
        flat = embed_with(m, [str(r[o]) for _, r in df.iterrows() for o in OPTIONS])
        return q.astype(np.float32), flat.reshape(len(df), len(OPTIONS), -1).astype(np.float32)

    qtr, otr = enc(train)
    qte, ote = enc(test)
    del m; gc.collect()

    # cosine(question, option) -- vectors are L2-normalised so this is a dot product
    s_tr = np.einsum("nd,nkd->nk", qtr, otr).astype(np.float32)
    s_te = np.einsum("nd,nkd->nk", qte, ote).astype(np.float32)
    return dict(dim=d, q_train=qtr, o_train=otr, q_test=qte, o_test=ote,
                s_train=s_tr, s_test=s_te)


val_pos = train.index[train["prompt"].isin(va_p)].to_numpy()

for name in ENCODERS:
    try:
        ENC[name] = run_encoder(name)
        short = name.split("/")[-1]
        evaluate(f"2. {short}", y_val, ENC[name]["s_train"][val_pos], RESULTS)
    except Exception as e:
        print(f"  skipped ({type(e).__name__}: {str(e)[:90]})")

assert ENC, "no encoder could be loaded"

# ---- combine encoders: average of within-question z-scored cosines ----
def zrow(a):
    a = np.asarray(a, dtype=np.float64)
    sd = a.std(axis=1, keepdims=True)
    return (a - a.mean(axis=1, keepdims=True)) / np.where(sd > 1e-9, sd, 1.0)


mpnet_train = np.mean([zrow(v["s_train"]) for v in ENC.values()], axis=0).astype(np.float32)
mpnet_test  = np.mean([zrow(v["s_test"])  for v in ENC.values()], axis=0).astype(np.float32)
mpnet_val   = mpnet_train[val_pos]

evaluate(f"2. ENCODER BLEND ({len(ENC)})", y_val, mpnet_val, RESULTS)
evaluate("   blend on all 2000 rows", y_train, mpnet_train)
print("   ^ agree because no training corpus is used")

# downstream heads use the strongest single encoder's embeddings
best_enc = max(ENC, key=lambda n: map_at_3(y_val, top3(ENC[n]["s_train"][val_pos])))
print(f"\nStrongest encoder: {best_enc.split('/')[-1]}")
Q_train, O_train = ENC[best_enc]["q_train"], ENC[best_enc]["o_train"]
Q_test,  O_test  = ENC[best_enc]["q_test"],  ENC[best_enc]["o_test"]
EMB_DIM = ENC[best_enc]["dim"]
Q_val, O_val = Q_train[val_pos], O_train[val_pos]


## Model 3 — Deepnet over MPNet pair features  *(supervised)*

```
  [ q ; o ; q*o ; |q-o| ]  (4 x 768)
            |
   Linear(3072 -> 256) + ReLU + Dropout
            |
   Linear(256 -> 64)   + ReLU + Dropout
            |
   Linear(64 -> 1)  ->  score per option
            |
   softmax over 5 options -> cross-entropy
```

The elementwise product and absolute difference are what let the head model
*matching* between question and option rather than the two vectors separately.
Trained 5-fold with `GroupKFold` on cleaned prompts, so every training row gets
an honest out-of-fold prediction.


In [ ]:
class DeepNet(nn.Module):
    def __init__(self, d, hidden=256, p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4 * d, hidden), nn.ReLU(), nn.Dropout(p),
            nn.Linear(hidden, 64), nn.ReLU(), nn.Dropout(p / 2),
            nn.Linear(64, 1))

    def forward(self, q, o):                       # q,o: (B,5,D)
        return self.net(torch.cat([q, o, q * o, (q - o).abs()], -1)).squeeze(-1)


def pair_inputs(Q, O):
    q = torch.tensor(np.repeat(Q[:, None, :], len(OPTIONS), axis=1))
    return q, torch.tensor(O)


def fit_deepnet(Q, O, y, epochs=30, lr=1e-3, bs=32, seed=SEED, d=None,
                hidden=256, drop=0.3):
    torch.manual_seed(seed)
    q, o = pair_inputs(Q, O)
    y = torch.tensor(y, dtype=torch.long)
    model = DeepNet(d or Q.shape[1], hidden=hidden, p=drop).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    q, o, y = q.to(DEVICE), o.to(DEVICE), y.to(DEVICE)

    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(y))
        for k in range(0, len(y), bs):
            b = perm[k:k + bs]
            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(q[b], o[b]), y[b])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
    return model


@torch.no_grad()
def deepnet_predict(model, Q, O):
    model.eval()
    q, o = pair_inputs(Q, O)
    return model(q.to(DEVICE), o.to(DEVICE)).cpu().numpy().astype(np.float32)


def run_folds(fit_fn, pred_fn, n_rows, n_test, tag):
    """5-fold OOF + averaged test predictions."""
    oof = np.zeros((n_rows, len(OPTIONS)), dtype=np.float32)
    tst = np.zeros((n_test, len(OPTIONS)), dtype=np.float32)
    gkf = GroupKFold(n_splits=N_FOLDS)
    groups = train["prompt"].values
    y_idx = np.array([OPTIONS.index(a) for a in train["answer"]])

    for f, (tr_i, va_i) in enumerate(gkf.split(train, groups=groups), 1):
        model = fit_fn(tr_i, y_idx[tr_i], SEED + f)
        oof[va_i] = pred_fn(model, va_i, "train")
        tst += pred_fn(model, None, "test") / N_FOLDS
        s = map_at_3([train["answer"].iloc[i] for i in va_i], top3(oof[va_i]))
        print(f"  {tag} fold {f}/{N_FOLDS}  holdout MAP@3 {s:.4f}")
        del model; gc.collect()
    return oof, tst


# --- small hyperparameter search on the validation split (cheap on CPU) ---
print("Tuning deepnet on validation...")
tr_mask = ~np.isin(np.arange(len(train)), val_pos)
tr_idx = np.where(tr_mask)[0]
y_idx_all = np.array([OPTIONS.index(a) for a in train["answer"]])

best_cfg, best_v = None, -np.inf
for hidden in (128, 256):
    for drop in (0.2, 0.4):
        for lr in (1e-3, 3e-3):
            m_ = fit_deepnet(Q_train[tr_idx], O_train[tr_idx], y_idx_all[tr_idx],
                             epochs=20, lr=lr, seed=SEED, d=EMB_DIM,
                             hidden=hidden, drop=drop)
            v = map_at_3(y_val, top3(deepnet_predict(m_, Q_val, O_val)))
            print(f"  hidden={hidden:<4} drop={drop:<4} lr={lr:<6} -> {v:.4f}")
            if v > best_v:
                best_v, best_cfg = v, dict(hidden=hidden, drop=drop, lr=lr)
            del m_; gc.collect()
print(f"best deepnet config: {best_cfg}  (val {best_v:.4f})")

print("Training Deepnet (best config), 5-fold...")
deep_oof, deep_test = run_folds(
    fit_fn=lambda idx, yy, sd: fit_deepnet(Q_train[idx], O_train[idx], yy, seed=sd,
                                       lr=best_cfg["lr"], d=EMB_DIM,
                                       hidden=best_cfg["hidden"],
                                       drop=best_cfg["drop"]),
    pred_fn=lambda m, idx, which: deepnet_predict(
        m, Q_train[idx] if which == "train" else Q_test,
        O_train[idx] if which == "train" else O_test),
    n_rows=len(train), n_test=len(test), tag="deepnet")

deep_val = deep_oof[val_pos]
evaluate("3. Deepnet (MPNet)", y_val, deep_val, RESULTS)
evaluate("   Deepnet OOF (2000)", y_train, deep_oof)


## Model 4 — TF-IDF + Deepnet  *(supervised)*

The same head, but over **TF-IDF** features instead of MPNet embeddings. The
sparse vectors are reduced with truncated SVD so the dense head can consume them.
This isolates how much of Model 3's performance comes from the neural head versus
from MPNet's semantics.


In [ ]:
from sklearn.decomposition import TruncatedSVD

SVD_DIM = 256
print(f"Reducing TF-IDF to {SVD_DIM} dims...")

all_q = tfidf.transform(pd.concat([train["prompt"], test["prompt"]]).astype(str))
all_o = tfidf.transform(
    [str(r[o]) for _, r in train.iterrows() for o in OPTIONS] +
    [str(r[o]) for _, r in test.iterrows() for o in OPTIONS])

svd = TruncatedSVD(n_components=SVD_DIM, random_state=SEED)
svd.fit(all_q)

def svd_q(df):
    return normalize(svd.transform(tfidf.transform(df["prompt"].astype(str)))).astype(np.float32)

def svd_o(df):
    m = svd.transform(tfidf.transform([str(r[o]) for _, r in df.iterrows() for o in OPTIONS]))
    return normalize(m).reshape(len(df), len(OPTIONS), -1).astype(np.float32)

Qt_train, Ot_train = svd_q(train), svd_o(train)
Qt_test,  Ot_test  = svd_q(test),  svd_o(test)
print(f"explained variance: {svd.explained_variance_ratio_.sum():.1%}")

print("Training TF-IDF + Deepnet, 5-fold...")
tdeep_oof, tdeep_test = run_folds(
    fit_fn=lambda idx, yy, sd: fit_deepnet(Qt_train[idx], Ot_train[idx], yy, seed=sd),
    pred_fn=lambda m, idx, which: deepnet_predict(
        m, Qt_train[idx] if which == "train" else Qt_test,
        Ot_train[idx] if which == "train" else Ot_test),
    n_rows=len(train), n_test=len(test), tag="tfidf-deep")

tdeep_val = tdeep_oof[val_pos]
evaluate("4. TF-IDF + Deepnet", y_val, tdeep_val, RESULTS)


## Model 5 — BiLSTM + Attention  *(model from scratch)*

No pre-trained weights anywhere: the word embedding table is learned from this
corpus alone.

```
  "question <sep> option"
          |
   Embedding(V, 128)          <- learned from scratch
          |
   BiLSTM(128 -> 2x128)
          |
   attention:  a = softmax(w.h) ,  pooled = sum(a * h)
          |
   Linear(256 -> 64) -> ReLU -> Linear(64 -> 1)
          |
   softmax over the 5 options -> cross-entropy
```

Attention pooling lets the model weight the tokens that decide the answer instead
of averaging the whole sequence.


In [ ]:
from collections import Counter

MAX_LEN, MIN_FREQ, EMB_SZ, HID = 96, 2, 128, 128
# Attention over a learned embedding table needs enough passes to converge;
# cutting this short was leaving the model undertrained.
BILSTM_EPOCHS = 30


def toks(s):
    return re.findall(r"[a-z0-9']+", str(s).lower())


cnt = Counter()
for _, r in train.iterrows():
    cnt.update(toks(r["prompt"]))
    for o in OPTIONS:
        cnt.update(toks(r[o]))

VOCAB = {"<pad>": 0, "<unk>": 1, "<sep>": 2}
for w, c in cnt.most_common():
    if c >= MIN_FREQ:
        VOCAB[w] = len(VOCAB)
print(f"vocabulary: {len(VOCAB)} types")


def encode_pair(p, o):
    ids = ([VOCAB.get(w, 1) for w in toks(p)][:MAX_LEN // 2] + [2] +
           [VOCAB.get(w, 1) for w in toks(o)])[:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))


def seq_ids(df):
    return torch.tensor(np.array(
        [[encode_pair(r["prompt"], r[o]) for o in OPTIONS] for _, r in df.iterrows()],
        dtype=np.int64))


IDS_train, IDS_test = seq_ids(train), seq_ids(test)


class BiLSTMAttn(nn.Module):
    def __init__(self, vocab, emb=EMB_SZ, hid=HID, p=0.3):
        super().__init__()
        self.emb  = nn.Embedding(vocab, emb, padding_idx=0)
        self.lstm = nn.LSTM(emb, hid, batch_first=True, bidirectional=True)
        self.attn = nn.Linear(2 * hid, 1)
        self.drop = nn.Dropout(p)
        self.head = nn.Sequential(nn.Linear(2 * hid, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, ids):                        # (B,5,L)
        B, K, L = ids.shape
        flat = ids.reshape(B * K, L)
        mask = flat != 0
        h, _ = self.lstm(self.emb(flat))
        a = self.attn(h).squeeze(-1).masked_fill(~mask, -1e9)
        pooled = self.drop((h * torch.softmax(a, -1).unsqueeze(-1)).sum(1))
        return self.head(pooled).reshape(B, K)


def fit_bilstm(idx, yy, seed):
    torch.manual_seed(seed)
    ids = IDS_train[idx].to(DEVICE)
    y = torch.tensor(yy, dtype=torch.long, device=DEVICE)
    model = BiLSTMAttn(len(VOCAB)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=BILSTM_EPOCHS)
    for ep in range(BILSTM_EPOCHS):
        model.train()
        perm = torch.randperm(len(y))
        for k in range(0, len(y), 32):
            b = perm[k:k + 32]
            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(ids[b]), y[b])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
    return model


@torch.no_grad()
def bilstm_predict(model, idx, which, bs=64):
    model.eval()
    ids = IDS_train[idx] if which == "train" else IDS_test
    out = [model(ids[k:k + bs].to(DEVICE)).cpu().numpy() for k in range(0, len(ids), bs)]
    return np.concatenate(out).astype(np.float32)


print(f"Training BiLSTM+Attention ({BILSTM_EPOCHS} epochs/fold), 5-fold...")
bl_oof, bl_test = run_folds(fit_bilstm, bilstm_predict, len(train), len(test),
                            "bilstm", n_seeds=2)

bl_val = bl_oof[val_pos]
evaluate("5. BiLSTM + Attention", y_val, bl_val, RESULTS)
evaluate("   BiLSTM OOF (2000)", y_train, bl_oof)


## Model 6 — DeBERTa-v3 fine-tune  *(additional experiment)*

Kept for completeness. Fine-tuning a 435M cross-encoder costs far more than the
other five models combined, so it runs only when an accelerator is available.

In the run where it did train, its loss never left `ln(5) = 1.6094` — the loss of
a uniform random 5-way classifier — so it scored below the 0.4213 prior baseline
and was excluded from the final submission. The block below prints a warning if
that happens again.


In [ ]:
if not RUN_DEBERTA:
    deberta_val = None
    print("DeBERTa skipped (no GPU).")
    print("On GPU it took ~2 h and its loss stayed at ln(5)=1.6094 — it never")
    print("learned, scoring 0.4062 OOF against a 0.4213 prior baseline.")
else:
    from transformers import AutoTokenizer, AutoModelForMultipleChoice
    from torch.utils.data import Dataset, DataLoader

    DEB_ID, MAXLEN, EPOCHS, LR = "microsoft/deberta-v3-base", 320, 3, 1.5e-5
    tok = AutoTokenizer.from_pretrained(DEB_ID)

    class DS(Dataset):
        def __init__(self, df): self.df = df.reset_index(drop=True)
        def __len__(self): return len(self.df)
        def __getitem__(self, i):
            r = self.df.iloc[i]
            e = tok([str(r["prompt"])] * 5, [str(r[o]) for o in OPTIONS],
                    truncation=True, max_length=MAXLEN, padding="max_length",
                    return_tensors="pt")
            d = dict(e)
            d["labels"] = torch.tensor(OPTIONS.index(r["answer"]))
            return d

    def collate(b): return {k: torch.stack([x[k] for x in b]) for k in b[0]}

    model = AutoModelForMultipleChoice.from_pretrained(DEB_ID).float().to(DEVICE)
    dl = DataLoader(DS(train_split), batch_size=4, shuffle=True, collate_fn=collate)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

    for ep in range(EPOCHS):
        model.train(); tot = 0.0
        for b in dl:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            opt.zero_grad(set_to_none=True)
            loss = model(**b).loss
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); tot += loss.item()
        avg = tot / len(dl)
        print(f"  epoch {ep+1}: loss {avg:.4f}"
              + ("   <-- still at ln(5): NOT learning" if abs(avg - math.log(5)) < 0.05 else ""))

    model.eval(); out = []
    with torch.no_grad():
        for b in DataLoader(DS(val_split), batch_size=8, collate_fn=collate):
            b = {k: v.to(DEVICE) for k, v in b.items()}
            out.append(model(**{k: v for k, v in b.items() if k != "labels"})
                       .logits.cpu().numpy())
    deberta_val = np.concatenate(out)
    evaluate("6. DeBERTa-v3", y_val, deberta_val, RESULTS)
    del model; gc.collect()


## 7. Comparison

In [ ]:
rows = []
for name, m in RESULTS.items():
    rows.append({"Model": name, **{k: round(v, 4) for k, v in m.items()}})
comp = pd.DataFrame(rows).sort_values("MAP@3", ascending=False).reset_index(drop=True)

print(f"{'':<30}{'MAP@3':>9}{'Acc':>9}{'MacroF1':>10}{'WgtF1':>9}")
print("-" * 67)
for _, r in comp.iterrows():
    print(f"{r['Model']:<30}{r['MAP@3']:>9.4f}{r['Accuracy']:>9.4f}"
          f"{r['MacroF1']:>10.4f}{r['WgtF1']:>9.4f}")
print("-" * 67)
print(f"{'random ordering':<30}{0.3667:>9.4f}")
print(f"{'label prior (B C A)':<30}{PRIOR_MAP3:>9.4f}")

best = comp.iloc[0]["Model"]
print(f"\nBest single model: {best}")
comp


## 8. Ensemble and submission

Each model's five scores are centred within the question and divided by that
model's global spread, then combined with weights chosen by a small random
search on the validation set. The label prior is included as a tie-breaker.

The search always includes the **best single model alone** as a candidate, so
the ensemble can never be selected unless it actually beats it.


In [ ]:
# ---------------------------------------------------------------------------
# Weights are fitted on ALL 2000 rows, not the 400-row validation slice.
#
# Honest scores already exist for every training row: tfidf and the encoder
# blend are unsupervised (no fitting, so valid everywhere), and the three heads
# have out-of-fold predictions from models that never saw the row they score.
#
# The standard error of a MAP@3 estimate is roughly 0.35/sqrt(n): +/-0.018 at
# n=400 versus +/-0.008 at n=2000. On 400 rows the search cannot tell apart two
# weightings that differ by less than the entire gap to the cut-off, so it fits
# noise. Five times the data more than halves that.
# ---------------------------------------------------------------------------
MODELS = {
    "tfidf":  (tfidf_train, tfidf_test),
    "mpnet":  (mpnet_train, mpnet_test),
    "deep":   (deep_oof,    deep_test),
    "tdeep":  (tdeep_oof,   tdeep_test),
    "bilstm": (bl_oof,      bl_test),
}
NAMES = list(MODELS)
FIT_Y = y_train                       # all 2000 rows


def centre(a):
    a = np.asarray(a, dtype=np.float64)
    return a - a.mean(axis=1, keepdims=True)


SCALES = {m: max(centre(MODELS[m][0]).std(), 1e-8) for m in NAMES}
PRIOR_C = centre(LABEL_LOGPRIOR[None, :])[0]
PRIOR_S = max(PRIOR_C.std(), 1e-8)


def blend(part, w):
    tot = np.zeros((len(MODELS[NAMES[0]][part]), len(OPTIONS)))
    for m in NAMES:
        if w.get(m):
            tot += w[m] * centre(MODELS[m][part]) / SCALES[m]
    if w.get("prior"):
        tot += w["prior"] * (PRIOR_C / PRIOR_S)[None, :]
    return tot


print("Per-model MAP@3 over all 2000 rows (out-of-fold where trained):")
for m in NAMES:
    print(f"  {m:<8} {map_at_3(FIT_Y, top3(MODELS[m][0])):.4f}")
print(f"  {'prior':<8} {PRIOR_MAP3:.4f}   (trivial baseline)")

rng = np.random.default_rng(SEED)
cands  = [{m: 1.0 for m in NAMES}]
cands += [{n: (1.0 if n == m else 0.0) for n in NAMES} for m in NAMES]   # each alone
cands += [{"mpnet": 3, "deep": 2, "bilstm": 2, "tdeep": 1, "tfidf": 1, "prior": 1},
          {"mpnet": 3, "deep": 3, "bilstm": 3, "prior": 1},
          {"deep": 3, "bilstm": 3, "tdeep": 2, "prior": 1}]
cands += [{m: float(rng.choice([0, 0.5, 1, 2, 3])) for m in NAMES} |
          {"prior": float(rng.choice([0, 0.5, 1]))} for _ in range(600)]

best_w, best_s = None, -np.inf
for w in cands:
    if sum(v for k, v in w.items() if k in NAMES) == 0:
        continue
    s = map_at_3(FIT_Y, top3(blend(0, w)))
    if s > best_s:
        best_s, best_w = s, w

single_best = max(NAMES, key=lambda m: map_at_3(FIT_Y, top3(MODELS[m][0])))
single_s = map_at_3(FIT_Y, top3(MODELS[single_best][0]))

print(f"\nbest blend      : {best_s:.4f}   { {k: v for k, v in best_w.items() if v} }")
print(f"best single     : {single_best} ({single_s:.4f})")

# hold-out sanity check: the same weights scored on the validation slice only
print(f"validation slice: blend {map_at_3(y_val, top3(blend(0, best_w)[val_pos])):.4f} | "
      f"{single_best} {map_at_3(y_val, top3(MODELS[single_best][0][val_pos])):.4f}")

if best_s > single_s + 1e-6:
    final_test, chosen = blend(1, best_w), "ensemble"
    val_final = blend(0, best_w)
else:
    final_test, chosen = MODELS[single_best][1], f"single model: {single_best}"
    val_final = MODELS[single_best][0]
print(f"\nUsing -> {chosen}")

print("\nPer-class report (all 2000 rows, chosen config):")
print(classification_report(FIT_Y, [p[0] for p in top3(val_final)],
                            labels=OPTIONS, zero_division=0))


In [ ]:
preds = top3(final_test)
submission = pd.DataFrame({"ID": test_df["id"], "Prediction": [" ".join(p) for p in preds]})

assert len(submission) == len(test_df)
assert submission["Prediction"].str.split().apply(len).eq(3).all()
assert submission["Prediction"].apply(
    lambda p: len(set(p.split())) == 3 and set(p.split()) <= set(OPTIONS)).all()

submission.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {OUTPUT_PATH}  ({len(submission)} rows)")
print(submission.head(10).to_string(index=False))
print("\nFirst-choice distribution:")
print(submission["Prediction"].str[0].value_counts().sort_index())
